# 02 Training

QLoRA fine-tuning on Kaggle T4 (see `docs/claude-3.md` Step 7). Local run requires a CUDA GPU and enough VRAM.

In [1]:
# ── Colab Setup ────────────────────────────────────────────
import os, torch

# 1. Verify GPU
assert torch.cuda.is_available(), "No GPU! Runtime → Change Runtime Type → T4"
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# 2. Clone repo into Colab working dir
if not os.path.exists('/content/jinyong-finetune'):
    !git clone https://github.com/jxjwilliam/jinyong-finetune.git /content/jinyong-finetune
%cd /content/jinyong-finetune

# 3. Install deps
!pip install -q \
    transformers==4.44.0 \
    peft==0.12.0 \
    trl==0.10.1 \
    accelerate==0.34.0 \
    bitsandbytes==0.43.3 \
    datasets==2.21.0 \
    pyyaml kaggle

print("Setup complete.")

GPU  : Tesla T4
VRAM : 15.6 GB
Cloning into '/content/jinyong-finetune'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 52 (delta 15), reused 46 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 36.26 KiB | 7.25 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/jinyong-finetune
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
# ── Download Jinyong dataset via Kaggle API ─────────────────

# First: upload your kaggle.json
from google.colab import files
print("Upload your kaggle.json now (from kaggle.com → Account → API → Create Token)")
files.upload()   # select kaggle.json from your local machine

# Move it to the right place
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset
!kaggle datasets download -d evilpsycho42/jinyong-wuxia \
    -p /content/jinyong-finetune/data/raw --unzip

!ls -lh /content/jinyong-finetune/data/raw/

Upload your kaggle.json now (from kaggle.com → Account → API → Create Token)


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/evilpsycho42/jinyong-wuxia
License(s): unknown
100% 10.8M/10.8M [00:01<00:00, 6.94MB/s]

total 25M
-rw-r--r-- 1 root root 4.7K Apr 30 18:48 cn_stopwords.txt
-rw-r--r-- 1 root root  25M Apr 30 18:48 jinyong.txt
-rw-r--r-- 1 root root  21K Apr 30 18:48 person_count.txt
-rw-r--r-- 1 root root  15K Apr 30 18:48 person.txt


In [6]:
# Verify all critical imports work — ignore gcsfs entirely
import torch
import transformers, peft, trl, accelerate, datasets
import yaml

print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"trl          : {trl.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"torch        : {torch.__version__}")
print(f"GPU ready    : {torch.cuda.get_device_name(0)}")

transformers : 4.44.0
peft         : 0.12.0
trl          : 0.10.1
datasets     : 2.21.0
torch        : 2.10.0+cu128
GPU ready    : Tesla T4


### The above the basic setup: bring .ipynb, git clone, set 'T4 GPU', pip install, download datasets

In [9]:
%cd /content/jinyong-finetune

/content/jinyong-finetune


In [10]:
!python scripts/clean_text.py --dry-run

[DRY RUN] cn_stopwords.txt               utf-8         2,079 →      2,078 chars (-0.0%)
[DRY RUN] jinyong.txt                    utf-8     8,699,641 →  5,895,221 chars (-32.2%)
[DRY RUN] person.txt                     utf-8         6,879 →      5,457 chars (-20.7%)
[DRY RUN] person_count.txt               utf-8        13,533 →      9,410 chars (-30.5%)

Total: 8,722,132 → 5,912,166 chars


In [11]:
# ── Clean + build instruction pairs ────────────────────────
%cd /content/jinyong-finetune

!python scripts/clean_text.py

!python scripts/build_instructions.py --stats

# Verify
!wc -l data/instructions/jinyong_sft.jsonl

/content/jinyong-finetune
cn_stopwords.txt               utf-8         2,079 →      2,078 chars (-0.0%)
jinyong.txt                    utf-8     8,699,641 →  5,895,221 chars (-32.2%)
person.txt                     utf-8         6,879 →      5,457 chars (-20.7%)
person_count.txt               utf-8        13,533 →      9,410 chars (-30.5%)

Total: 8,722,132 → 5,912,166 chars
cn_stopwords.txt: 8 sliding windows
jinyong.txt: 29474 sliding windows
person.txt: 25 sliding windows
person_count.txt: 45 sliding windows
Continuation pairs:  29,552
Typed scene pairs:   3,694
Total pairs:         33,246
Train / Val split:   31,584 / 1,662  (eval_ratio=0.05, seed=42)
Saved → data/instructions/jinyong_sft.jsonl
33246 data/instructions/jinyong_sft.jsonl


In [12]:
from pathlib import Path
import os


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts" / "train.py").is_file():
            return cand
    raise FileNotFoundError("Could not find scripts/train.py")


os.chdir(find_repo_root())
print("CWD:", Path.cwd())

CWD: /content/jinyong-finetune


## Kaggle: dependencies (uncomment)
```
# !pip install -q transformers==4.44.0 peft==0.12.0 trl==0.10.1 accelerate==0.34.0 bitsandbytes==0.43.3 datasets==2.21.0 pyyaml tqdm torch
# !git clone https://github.com/jxjwilliam/jinyong-finetune.git /kaggle/working/jinyong-finetune
# %cd /kaggle/working/jinyong-finetune
```

In [13]:
import torch

assert torch.cuda.is_available(), "No GPU — enable GPU in Kaggle Settings or use a CUDA machine"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (approx):", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

GPU: Tesla T4
VRAM (approx): 15.637086208 GB


## Train
Hyperparameters come only from `configs/qlora_config.yaml`.

In [15]:
import subprocess
import sys

try:
    result = subprocess.run(
        [sys.executable, "scripts/train.py", "--config", "configs/qlora_config.yaml"],
        check=True,
        capture_output=True,
        text=True
    )
    print("STDOUT:", result.stdout)
except subprocess.CalledProcessError as e:
    print(f"Error executing train.py: {e}")
    print(f"STDOUT:\n{e.stdout}")
    print(f"STDERR:\n{e.stderr}")

Error executing train.py: Command '['/usr/bin/python3', 'scripts/train.py', '--config', 'configs/qlora_config.yaml']' returned non-zero exit status 1.
STDOUT:

STDERR:
2026-04-30 19:00:58.164558: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/content/jinyong-finetune/scripts/train.py", line 135, in <module>
    main()
  File "/content/jinyong-finetune/scripts/train.py", line 48, in main
    compute_dtype = dtype_map[model_cfg["bnb_4bit_compute_dtype"]]
                              ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyError: 'bnb_4bit_compute_dtype'



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## After training
Adapter weights are written under `outputs/jinyong-qlora/adapter/` (see `training.output_dir` in the YAML). On Kaggle, copy that folder to `/kaggle/working/` if you need it in the session output.